# 🏠 House Prices — Advanced Regression Techniques

Predict sales prices using feature engineering and advanced regression.

**Competition metric**: RMSE on log-transformed SalePrice (RMSLE)

This notebook covers:
1. 📊 Exploratory Data Analysis
2. 🔧 Competition-grade Feature Engineering
3. 🤖 Baseline Models (CART, Random Forest, Gradient Boosting, XGBoost)
4. 🚀 Full-power Hyperparameter Tuning (resumable)
5. 🏆 Ensembling & Final Submission

---
# ⚙️ Phase 0: Setup & Data Loading

Configure the environment toggle and import all required libraries.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION — Change KAGGLE to True when running on Kaggle
# ══════════════════════════════════════════════════════════════
KAGGLE = True

if KAGGLE:
    DATA_DIR = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/'
    OUTPUT_DIR = '/kaggle/working/'
else:
    DATA_DIR = './data/'
    OUTPUT_DIR = './'

print(f'Data directory: {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import boxcox1p
from scipy.stats import skew, norm, probplot

from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor,
    VotingRegressor,
)
from xgboost import XGBRegressor

from tqdm.notebook import tqdm

import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('All libraries loaded successfully ✅')

### Load the data

In [ ]:
train = pd.read_csv(DATA_DIR + 'train.csv')
test = pd.read_csv(DATA_DIR + 'test.csv')

# Save test IDs for submission
test_ids = test['Id']

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Target (SalePrice) range: ${train["SalePrice"].min():,.0f} – ${train["SalePrice"].max():,.0f}')
print(f'Target mean: ${train["SalePrice"].mean():,.0f}, median: ${train["SalePrice"].median():,.0f}')
train.head()

---
# 📊 Phase 1: Exploratory Data Analysis (EDA)

Before building models, we need to deeply understand our data — its distributions, relationships, missing patterns, and outliers. This informs every decision in feature engineering and modeling.

## 1.1 Target Variable: SalePrice

The competition evaluates on **RMSLE** (Root Mean Squared Log Error), which means we need to understand how SalePrice behaves in both raw and log-transformed space.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Raw distribution
sns.histplot(train['SalePrice'], kde=True, ax=axes[0, 0], color='#3498db', bins=50)
axes[0, 0].set_title('SalePrice Distribution (Raw)')
axes[0, 0].axvline(train['SalePrice'].mean(), color='red', linestyle='--', label=f'Mean: ${train["SalePrice"].mean():,.0f}')
axes[0, 0].axvline(train['SalePrice'].median(), color='green', linestyle='--', label=f'Median: ${train["SalePrice"].median():,.0f}')
axes[0, 0].legend()

# Log-transformed distribution
log_price = np.log1p(train['SalePrice'])
sns.histplot(log_price, kde=True, ax=axes[0, 1], color='#2ecc71', bins=50)
axes[0, 1].set_title('SalePrice Distribution (Log-Transformed)')

# Q-Q plot (raw)
probplot(train['SalePrice'], dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Raw)')

# Q-Q plot (log)
probplot(log_price, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot (Log-Transformed)')

plt.tight_layout()
plt.show()

print(f'Skewness (raw):  {train["SalePrice"].skew():.4f}')
print(f'Kurtosis (raw):  {train["SalePrice"].kurt():.4f}')
print(f'Skewness (log):  {log_price.skew():.4f}')
print(f'Kurtosis (log):  {log_price.kurt():.4f}')
print()
print('💡 The log-transformed SalePrice is much closer to a normal distribution.')
print('   This is crucial because our metric (RMSLE) operates in log-space.')

## 1.2 Missing Values Analysis

Understanding *why* data is missing is as important as knowing *what* is missing. In this dataset, many 'missing' values actually mean 'feature not present' (e.g., no pool, no garage, no basement).

In [ ]:
# Calculate missing percentages for both sets
train_missing = train.isnull().sum()
train_missing = train_missing[train_missing > 0].sort_values(ascending=False)
train_missing_pct = (train_missing / len(train) * 100).round(1)

test_missing = test.isnull().sum()
test_missing = test_missing[test_missing > 0].sort_values(ascending=False)
test_missing_pct = (test_missing / len(test) * 100).round(1)

# Combine into a single DataFrame
missing_df = pd.DataFrame({
    'Train Count': train_missing,
    'Train %': train_missing_pct,
}).join(pd.DataFrame({
    'Test Count': test_missing,
    'Test %': test_missing_pct,
}), how='outer').fillna(0).sort_values('Train %', ascending=False)

print(f'Features with missing values in train: {len(train_missing)}')
print(f'Features with missing values in test:  {len(test_missing)}')
print()
missing_df

In [ ]:
# Visual missing-value heatmap
cols_with_missing = train.columns[train.isnull().any()].tolist()
if cols_with_missing:
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(train[cols_with_missing].isnull().T, cbar=True, cmap='YlOrRd',
                yticklabels=True, ax=ax)
    ax.set_title('Missing Values Heatmap (Train Set)', fontsize=14)
    ax.set_xlabel('Sample Index')
    plt.tight_layout()
    plt.show()

print('\n💡 Key observations:')
print('  • PoolQC, MiscFeature, Alley, Fence: >80% missing — these mean "None" (no pool/alley/fence)')
print('  • FireplaceQu: 47% missing — means no fireplace')
print('  • Garage features: 5.5% missing together — means no garage')
print('  • Basement features: 2.5% missing together — means no basement')
print('  • LotFrontage: 17.7% — will impute by neighborhood median')

## 1.3 Correlation Analysis

Which numeric features are most strongly correlated with SalePrice? This helps us identify the most predictive features and spot multicollinearity.

In [ ]:
# Top-20 correlations with SalePrice
numeric_cols = train.select_dtypes(include=[np.number]).columns
correlations = train[numeric_cols].corr()['SalePrice'].drop('SalePrice').abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Bar chart of top-20
top20 = correlations.head(20)
colors = ['#e74c3c' if v > 0.5 else '#f39c12' if v > 0.3 else '#3498db' for v in top20.values]
top20.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('Top 20 Features Correlated with SalePrice')
axes[0].set_xlabel('|Correlation|')
axes[0].invert_yaxis()

# Heatmap of top-12 correlated features
top12_cols = correlations.head(12).index.tolist() + ['SalePrice']
sns.heatmap(train[top12_cols].corr(), annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, ax=axes[1], square=True, linewidths=0.5)
axes[1].set_title('Correlation Heatmap (Top 12 + SalePrice)')

plt.tight_layout()
plt.show()

print('💡 Strongest predictors:')
for feat, corr in correlations.head(10).items():
    print(f'  {feat:20s}  r = {corr:.3f}')

## 1.4 Key Feature Deep-Dives

Let's examine the most important features in detail.

### 1.4.1 OverallQual — The #1 Predictor

Overall material and finish quality (1-10 scale). This is consistently the strongest single predictor of house price.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(x='OverallQual', y='SalePrice', data=train, ax=axes[0],
            palette='viridis')
axes[0].set_title('SalePrice by OverallQual')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Scatter with trend
qual_medians = train.groupby('OverallQual')['SalePrice'].median()
axes[1].bar(qual_medians.index, qual_medians.values, color='#3498db', alpha=0.7)
axes[1].set_title('Median SalePrice by OverallQual')
axes[1].set_xlabel('OverallQual')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()
print(f'Correlation: {train["OverallQual"].corr(train["SalePrice"]):.3f}')

### 1.4.2 GrLivArea — Above Grade Living Area & Outlier Detection

GrLivArea is the 2nd strongest predictor. The scatter plot reveals **2 famous outliers**: houses with >4,000 sq ft but very low prices. These are well-documented anomalies that should be removed before modeling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before outlier highlight
axes[0].scatter(train['GrLivArea'], train['SalePrice'], alpha=0.5, c='#3498db', s=20)
# Highlight the 2 outliers
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)]
axes[0].scatter(outliers['GrLivArea'], outliers['SalePrice'], c='red', s=100,
                marker='X', zorder=5, label=f'Outliers ({len(outliers)} points)')
axes[0].set_xlabel('GrLivArea (sq ft)')
axes[0].set_ylabel('SalePrice ($)')
axes[0].set_title('GrLivArea vs SalePrice — Outliers Highlighted')
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Log-space scatter
axes[1].scatter(train['GrLivArea'], np.log1p(train['SalePrice']), alpha=0.5, c='#2ecc71', s=20)
axes[1].scatter(outliers['GrLivArea'], np.log1p(outliers['SalePrice']), c='red', s=100,
                marker='X', zorder=5)
axes[1].set_xlabel('GrLivArea (sq ft)')
axes[1].set_ylabel('log(SalePrice)')
axes[1].set_title('GrLivArea vs log(SalePrice)')

plt.tight_layout()
plt.show()

print(f'⚠️ Found {len(outliers)} outliers with GrLivArea > 4000 and SalePrice < $300K')
print(f'   These will be removed in Phase 2 (Feature Engineering).')
print(f'   Outlier IDs: {outliers["Id"].values}')

### 1.4.3 Neighborhood — Location, Location, Location

Neighborhood captures location effects, which are among the strongest determinants of house prices.

In [ ]:
# Order neighborhoods by median price
neighborhood_order = train.groupby('Neighborhood')['SalePrice'].median().sort_values().index

fig, ax = plt.subplots(figsize=(16, 6))
sns.boxplot(x='Neighborhood', y='SalePrice', data=train, order=neighborhood_order,
            palette='viridis', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('SalePrice Distribution by Neighborhood')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

print('💡 Price ranges from ~$90K (MeadowV, BrDale) to ~$335K (NoRidge, NridgHt, StoneBr)')

### 1.4.4 Year Effects — Age Matters

Newer houses and recently remodeled houses tend to sell for more.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# YearBuilt
axes[0].scatter(train['YearBuilt'], train['SalePrice'], alpha=0.3, c='#3498db', s=15)
year_means = train.groupby('YearBuilt')['SalePrice'].mean()
axes[0].plot(year_means.index, year_means.values, color='red', linewidth=2, label='Mean')
axes[0].set_xlabel('Year Built')
axes[0].set_ylabel('SalePrice')
axes[0].set_title('SalePrice vs Year Built')
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# YearRemodAdd
axes[1].scatter(train['YearRemodAdd'], train['SalePrice'], alpha=0.3, c='#e74c3c', s=15)
remod_means = train.groupby('YearRemodAdd')['SalePrice'].mean()
axes[1].plot(remod_means.index, remod_means.values, color='blue', linewidth=2, label='Mean')
axes[1].set_xlabel('Year Remodeled')
axes[1].set_ylabel('SalePrice')
axes[1].set_title('SalePrice vs Year Remodeled')
axes[1].legend()
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

### 1.4.5 Garage & Basement — Size Relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Garage
sns.boxplot(x='GarageCars', y='SalePrice', data=train, ax=axes[0], palette='Set2')
axes[0].set_title('SalePrice by Garage Capacity (Cars)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Basement
axes[1].scatter(train['TotalBsmtSF'], train['SalePrice'], alpha=0.4, c='#9b59b6', s=15)
axes[1].set_xlabel('Total Basement SF')
axes[1].set_ylabel('SalePrice')
axes[1].set_title('SalePrice vs Total Basement Area')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

## 1.5 Numeric Feature Skewness

Many numeric features are heavily right-skewed. We'll need to apply log/Box-Cox transformations to normalize them before modeling — this significantly helps linear models and can improve tree models too.

In [ ]:
# Compute skewness for all numeric features
numeric_feats = train.select_dtypes(include=[np.number]).columns.drop(['Id', 'SalePrice'])
skewed = train[numeric_feats].apply(lambda x: x.dropna().skew()).sort_values(ascending=False)
skewed_high = skewed[skewed.abs() > 0.75]

fig, ax = plt.subplots(figsize=(10, max(6, len(skewed_high) * 0.3)))
colors = ['#e74c3c' if v > 2 else '#f39c12' if v > 1 else '#3498db' for v in skewed_high.values]
skewed_high.plot(kind='barh', ax=ax, color=colors)
ax.set_title(f'Skewed Numeric Features (|skewness| > 0.75): {len(skewed_high)} features')
ax.set_xlabel('Skewness')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print(f'\n💡 {len(skewed_high)} features have |skewness| > 0.75')
print(f'   These will be Box-Cox transformed in Phase 2.')

## 1.6 EDA Summary

**Key takeaways for feature engineering:**

1. **Target**: SalePrice is right-skewed → use `log1p` transformation
2. **Outliers**: 2 houses with GrLivArea >4000 and low prices → remove
3. **Missing values**: Most are 'None' (no feature present), not truly missing
4. **Top predictors**: OverallQual, GrLivArea, GarageCars, GarageArea, TotalBsmtSF
5. **Multicollinearity**: GarageCars/GarageArea, TotalBsmtSF/1stFlrSF, TotRmsAbvGrd/GrLivArea
6. **Skewness**: Many features need transformation
7. **Neighborhood**: Strong location effect worth careful encoding

---
# 🔧 Phase 2: Feature Engineering & Preprocessing

Competition-grade feature engineering: outlier removal, missing value imputation, new feature creation, ordinal encoding, skewness correction, and one-hot encoding.

We process train and test **together** to ensure consistent encoding.

## 2.1 Outlier Removal

Remove the 2 well-documented outliers identified in EDA: houses with GrLivArea > 4000 sq ft but suspiciously low prices.

In [ ]:
print(f'Train shape before outlier removal: {train.shape}')
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]
print(f'Train shape after outlier removal:  {train.shape}')
print(f'Removed {1460 - len(train)} outliers')

## 2.2 Target Transformation

Since the competition metric is RMSLE, we train on `log1p(SalePrice)`. This way, our model's MSE in training directly optimizes the competition metric.

In [ ]:
# Separate target and transform
y_train = np.log1p(train['SalePrice'])

# Combine train and test for consistent preprocessing
train_rows = len(train)
all_data = pd.concat([train.drop('SalePrice', axis=1), test], axis=0, ignore_index=True)
all_data.drop('Id', axis=1, inplace=True)

print(f'Combined dataset shape: {all_data.shape}')
print(f'Train rows: {train_rows}, Test rows: {len(all_data) - train_rows}')

## 2.3 Missing Value Imputation

Context-aware imputation based on the data description. Most 'missing' values in this dataset actually indicate **absence of a feature** (no pool, no garage, etc.), not truly missing data.

In [ ]:
# ── Group 1: 'NA' means 'None' (feature not present) ──
# These are categorical features where NA means the house doesn't have that feature
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

# Numeric features where NA means 0 (no feature = 0 area/count)
zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath',
    'MasVnrArea'
]
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

print('✅ Filled "None" features (no pool/garage/basement/etc.)')

In [ ]:
# ── Group 2: LotFrontage — impute by neighborhood median ──
# Lot frontage tends to be similar within neighborhoods
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
print('✅ LotFrontage imputed by neighborhood median')

# ── Group 3: Mode imputation for rare missing categoricals ──
mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd',
             'Functional', 'SaleType', 'Utilities']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])
print('✅ Rare missing categoricals filled with mode')

# Verify no missing values remain
remaining = all_data.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) == 0:
    print('\n🎉 All missing values handled! Zero NaN remaining.')
else:
    print(f'\n⚠️ Remaining missing values:')
    print(remaining)

## 2.4 Feature Creation

Create new features that capture domain knowledge about house pricing. These engineered features often have stronger predictive power than raw features.

In [ ]:
# ── Area features ──
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBathrooms'] = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
                              all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] + all_data['EnclosedPorch'] +
                            all_data['3SsnPorch'] + all_data['ScreenPorch'] +
                            all_data['WoodDeckSF'])

# ── Age features ──
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# ── Binary flags ──
all_data['IsRemodeled'] = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)
all_data['IsNew'] = (all_data['YrSold'] == all_data['YearBuilt']).astype(int)
all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(int)
all_data['Has2ndFlr'] = (all_data['2ndFlrSF'] > 0).astype(int)

# ── Interaction features ──
all_data['QualCondProduct'] = all_data['OverallQual'] * all_data['OverallCond']
all_data['GarageInteraction'] = all_data['GarageCars'] * all_data['GarageArea']

new_features = ['TotalSF', 'TotalBathrooms', 'TotalPorchSF', 'HouseAge', 'RemodAge',
                'IsRemodeled', 'IsNew', 'HasPool', 'HasGarage', 'HasBsmt',
                'HasFireplace', 'Has2ndFlr', 'QualCondProduct', 'GarageInteraction']
print(f'✅ Created {len(new_features)} new features:')
for f in new_features:
    print(f'   • {f}')

## 2.5 Ordinal Encoding

Quality and condition features have a natural ordering (Ex > Gd > TA > Fa > Po). We encode these as ordinal integers to preserve this ordering information.

In [ ]:
# Define ordinal mappings
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
exposure_map = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
fintype_map = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
garage_finish_map = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
fence_map = {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
functional_map = {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8}

ordinal_mappings = {
    'ExterQual': quality_map,
    'ExterCond': quality_map,
    'BsmtQual': quality_map,
    'BsmtCond': quality_map,
    'HeatingQC': quality_map,
    'KitchenQual': quality_map,
    'FireplaceQu': quality_map,
    'GarageQual': quality_map,
    'GarageCond': quality_map,
    'PoolQC': quality_map,
    'BsmtExposure': exposure_map,
    'BsmtFinType1': fintype_map,
    'BsmtFinType2': fintype_map,
    'GarageFinish': garage_finish_map,
    'Fence': fence_map,
    'Functional': functional_map,
}

for col, mapping in ordinal_mappings.items():
    all_data[col] = all_data[col].map(mapping).fillna(0).astype(int)

print(f'✅ Ordinal-encoded {len(ordinal_mappings)} quality/condition features')

## 2.6 Type Corrections

MSSubClass is coded as a number but it's actually a categorical label (building class type). Convert it to string for proper one-hot encoding.

In [ ]:
# MSSubClass is really categorical despite being numeric
all_data['MSSubClass'] = all_data['MSSubClass'].astype(str)

# Also convert MoSold and YrSold to categorical (month/year are cyclical, not continuous)
all_data['MoSold'] = all_data['MoSold'].astype(str)
all_data['YrSold'] = all_data['YrSold'].astype(str)

print('✅ Converted MSSubClass, MoSold, YrSold to categorical')

## 2.7 Skewness Correction

Apply Box-Cox transformation to numeric features with high skewness (>0.75). This normalizes the distributions, which helps models learn better.

In [ ]:
# Get numeric features (after ordinal encoding, before one-hot)
numeric_feats = all_data.select_dtypes(include=[np.number]).columns.tolist()
skewness = all_data[numeric_feats].apply(lambda x: x.dropna().skew()).sort_values(ascending=False)
skewed_feats = skewness[skewness.abs() > 0.75].index.tolist()

print(f'Features with |skewness| > 0.75: {len(skewed_feats)}')

# Apply Box-Cox transformation (using λ=0.15 which works well for house prices)
lam = 0.15
for feat in skewed_feats:
    all_data[feat] = boxcox1p(all_data[feat], lam)

print(f'✅ Box-Cox (λ={lam}) applied to {len(skewed_feats)} skewed features')

## 2.8 One-Hot Encoding

Convert remaining nominal categorical features to dummy variables. This creates a binary column for each category.

In [ ]:
# One-hot encode remaining categorical columns
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns to one-hot encode: {len(cat_cols)}')

all_data = pd.get_dummies(all_data, columns=cat_cols, drop_first=False)

print(f'\nFinal dataset shape: {all_data.shape}')
print(f'  Total features: {all_data.shape[1]}')

## 2.9 Split Back to Train/Test

Separate the combined dataset back into training and test sets.

In [ ]:
X_train = all_data.iloc[:train_rows].values
X_test = all_data.iloc[train_rows:].values
feature_names = all_data.columns.tolist()

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'\n✅ Feature engineering complete! Ready for modeling.')

---
# 🤖 Phase 3: Baseline Models & Cross-Validation

Train all 4 model families with default hyperparameters and evaluate using 10-fold cross-validation. The metric is **RMSLE** (Root Mean Squared Log Error), which we compute as RMSE on the log-transformed target.

Since we already log-transformed `y_train`, we simply compute RMSE on predictions.

## Model Families

| # | Model | Implementation | Description |
|---|-------|---------------|-------------|
| 1 | **CART** | `DecisionTreeRegressor` | Single decision tree (CART algorithm) |
| 2 | **Random Forest** | `RandomForestRegressor` | Bagged ensemble of CART trees |
| 3 | **Gradient Boosted Trees** | `GradientBoostingRegressor` | Sequential boosting |
| 4 | **Distributed Gradient Boosted Trees** | `XGBRegressor` | Optimized distributed GB |

In [ ]:
# Define cross-validation strategy
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# RMSLE scorer (since y is already log-transformed, RMSE on y = RMSLE on original)
def rmsle_cv(model, X, y):
    """Compute RMSLE via 10-fold CV (using neg_MSE on log-transformed target)."""
    scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=kf)
    rmse_scores = np.sqrt(-scores)
    return rmse_scores

print('Cross-validation setup: 10-fold, shuffle=True, random_state=42')

In [ ]:
# ── Define all 4 baseline models ──
baseline_models = {
    'CART (Decision Tree)': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosted Trees': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost (Distributed GB)': XGBRegressor(
        n_estimators=100, random_state=42, n_jobs=-1,
        tree_method='hist', verbosity=0
    ),
}

# ── Evaluate each model ──
baseline_results = {}

print('=' * 65)
print(f'{"Model":35s} {"Mean RMSLE":>12s} {"Std":>10s}')
print('=' * 65)

for name, model in baseline_models.items():
    scores = rmsle_cv(model, X_train, y_train)
    baseline_results[name] = {'mean': scores.mean(), 'std': scores.std(), 'scores': scores}
    print(f'{name:35s} {scores.mean():12.5f} {scores.std():10.5f}')

print('=' * 65)
best_name = min(baseline_results, key=lambda k: baseline_results[k]['mean'])
print(f'\n🏆 Best baseline: {best_name} (RMSLE: {baseline_results[best_name]["mean"]:.5f})')

In [ ]:
# Visualize baseline results
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
names = list(baseline_results.keys())
means = [baseline_results[n]['mean'] for n in names]
stds = [baseline_results[n]['std'] for n in names]
colors = ['#2ecc71' if n == best_name else '#3498db' for n in names]

bars = axes[0].barh(names, means, xerr=stds, color=colors, capsize=5, alpha=0.85)
axes[0].set_xlabel('RMSLE (lower is better)')
axes[0].set_title('Baseline Model Comparison')
axes[0].invert_yaxis()
for bar, mean in zip(bars, means):
    axes[0].text(mean + 0.002, bar.get_y() + bar.get_height()/2,
                f'{mean:.4f}', va='center', fontweight='bold')

# Box plot of CV folds
fold_data = [baseline_results[n]['scores'] for n in names]
bp = axes[1].boxplot(fold_data, labels=[n.split(' (')[0] for n in names],
                     patch_artist=True, vert=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('RMSLE')
axes[1].set_title('CV Fold Distribution')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print('\n💡 Tree ensembles (RF, GBR, XGBoost) significantly outperform single CART.')
print('   Next: hyperparameter tuning to push these scores further!')

---
# 🚀 Phase 4: Full-Power Hyperparameter Tuning

Focused grid search across all 4 model families with:

- **5-fold cross-validation** for tuning (10-fold reserved for final evaluation)
- **Pickle-based checkpointing** — saves progress after every parameter combination
- **Granular tqdm progress bar** — per-combination tracking with live best-score display
- **Auto-resume** — if Kaggle times out, just re-run this cell to continue

> ⚡ Estimated runtime: ~3-5 hours on Kaggle. If it times out, simply re-run — it picks up exactly where it stopped.

### Hyperparameter Search Spaces

| Model | Key Parameters | Combos | × 5 Folds |
|-------|---------------|--------|-----------|
| CART | max_depth, min_samples_split/leaf, max_features, ccp_alpha | 72 | 360 fits |
| Random Forest | n_estimators, max_depth, min_samples_split/leaf, max_features | 72 | 360 fits |
| Gradient Boosted Trees | n_estimators, learning_rate, max_depth, subsample | 54 | 270 fits |
| XGBoost | n_estimators, learning_rate, max_depth, subsample, colsample_bytree, reg_alpha/lambda | 216 | 1,080 fits |
| **Total** | | **414** | **2,070 fits** |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RESUMABLE HYPERPARAMETER TUNING WITH GRANULAR PROGRESS TRACKING
# ══════════════════════════════════════════════════════════════════

from sklearn.model_selection import ParameterGrid
from sklearn.base import clone
import hashlib

CHECKPOINT_FILE = OUTPUT_DIR + 'tuning_checkpoint.pkl'

# ── Define all model tuning configurations ──
tuning_configs = {
    'CART (Decision Tree)': {
        'model': DecisionTreeRegressor(random_state=42),
        'params': {
            'max_depth': [5, 10, 15, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 5, 10],
            'max_features': ['sqrt', None],
            'ccp_alpha': [0.0, 0.01],
        },
    },
    'Random Forest': {
        'model': RandomForestRegressor(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 15, None],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2],
            'max_features': ['sqrt', 0.3],
        },
    },
    'Gradient Boosted Trees': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [3, 4, 5],
            'subsample': [0.8, 1.0],
        },
    },
    'XGBoost (Distributed GB)': {
        'model': XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist', verbosity=0),
        'params': {
            'n_estimators': [100, 300, 500],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [3, 4, 5],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0],
            'reg_alpha': [0, 0.1],
            'reg_lambda': [1, 5],
        },
    },
}

# ── Helper: create a hash for a parameter combo ──
def param_hash(model_name, params):
    key = f'{model_name}|{sorted(params.items())}'
    return hashlib.md5(key.encode()).hexdigest()

# ── Load or initialize checkpoint ──
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'rb') as f:
        checkpoint = pickle.load(f)
    print(f'📂 Loaded checkpoint: {len(checkpoint["evaluated"])} combos already evaluated')
else:
    checkpoint = {
        'evaluated': set(),           # set of param_hash strings
        'model_results': {},           # {model_name: {'best_score': float, 'best_params': dict}}
        'completed_models': set(),     # fully completed model names
    }
    print('🆕 Starting fresh — no checkpoint found')

# ── Count total parameter combinations ──
total_combos = sum(
    len(list(ParameterGrid(config['params'])))
    for config in tuning_configs.values()
)
already_done = len(checkpoint['evaluated'])
print(f'Total parameter combinations: {total_combos}')
print(f'Already evaluated: {already_done}')
print(f'Remaining: {total_combos - already_done}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# MAIN TUNING LOOP — Resumable with per-combo progress
# ══════════════════════════════════════════════════════════════

tuned_models = {}
kf_tune = KFold(n_splits=5, shuffle=True, random_state=42)

# Global progress bar across ALL models and combos
pbar = tqdm(total=total_combos, initial=already_done,
            desc='🔍 Tuning progress', unit='combo')

for model_name, config in tuning_configs.items():
    base_model = config['model']
    param_grid = list(ParameterGrid(config['params']))
    n_combos = len(param_grid)

    # Skip fully completed models
    if model_name in checkpoint['completed_models']:
        print(f'[SKIP] {model_name} — already tuned ✅')
        # Restore the best model
        best_info = checkpoint['model_results'][model_name]
        best_model = clone(base_model)
        best_model.set_params(**best_info['best_params'])
        tuned_models[model_name] = {
            'model': best_model,
            'best_score': best_info['best_score'],
            'best_params': best_info['best_params'],
        }
        pbar.update(n_combos - sum(
            1 for p in param_grid if param_hash(model_name, p) in checkpoint['evaluated']
        ))
        continue

    print(f'\n--- Tuning {model_name} ({n_combos} combinations × 5 folds) ---')

    # Initialize or restore running best for this model
    if model_name in checkpoint['model_results']:
        best_score = checkpoint['model_results'][model_name]['best_score']
        best_params = checkpoint['model_results'][model_name]['best_params']
    else:
        best_score = float('inf')
        best_params = None

    for params in param_grid:
        ph = param_hash(model_name, params)

        # Skip already evaluated combos
        if ph in checkpoint['evaluated']:
            pbar.update(0)  # Don't double-count
            continue

        # Evaluate this combo
        model = clone(base_model)
        model.set_params(**params)
        scores = cross_val_score(model, X_train, y_train,
                                scoring='neg_mean_squared_error', cv=kf_tune)
        rmse = np.sqrt(-scores).mean()

        if rmse < best_score:
            best_score = rmse
            best_params = params

        # Update checkpoint
        checkpoint['evaluated'].add(ph)
        checkpoint['model_results'][model_name] = {
            'best_score': best_score,
            'best_params': best_params,
        }

        # Save checkpoint every 10 combos
        if len(checkpoint['evaluated']) % 10 == 0:
            with open(CHECKPOINT_FILE, 'wb') as f:
                pickle.dump(checkpoint, f)

        pbar.update(1)
        pbar.set_postfix_str(f'{model_name} | best: {best_score:.5f}')

    # Model fully completed
    checkpoint['completed_models'].add(model_name)
    checkpoint['model_results'][model_name] = {
        'best_score': best_score,
        'best_params': best_params,
    }

    # Save checkpoint
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(checkpoint, f)

    # Store tuned model
    best_model = clone(base_model)
    best_model.set_params(**best_params)
    tuned_models[model_name] = {
        'model': best_model,
        'best_score': best_score,
        'best_params': best_params,
    }

    print(f'  ✅ {model_name}: RMSLE = {best_score:.5f}')
    print(f'     Best params: {best_params}')

pbar.close()
print('\n🎉 All models tuned!')

### Tuning Results Summary

In [ ]:
# ── Display tuning results ──
print('=' * 75)
print(f'{"Model":35s} {"Baseline RMSLE":>15s} {"Tuned RMSLE":>13s} {"Improvement":>12s}')
print('=' * 75)

for name in tuning_configs.keys():
    baseline = baseline_results[name]['mean']
    tuned = tuned_models[name]['best_score']
    improvement = baseline - tuned
    arrow = '⬇️' if improvement > 0 else '⬆️'
    print(f'{name:35s} {baseline:15.5f} {tuned:13.5f} {arrow} {abs(improvement):.5f}')

print('=' * 75)
best_tuned = min(tuned_models, key=lambda k: tuned_models[k]['best_score'])
print(f'\n🏆 Best tuned model: {best_tuned} (RMSLE: {tuned_models[best_tuned]["best_score"]:.5f})')

# Show best params for each
print('\n📋 Best hyperparameters:')
for name, info in tuned_models.items():
    print(f'\n  {name}:')
    for k, v in info['best_params'].items():
        print(f'    {k}: {v}')

In [ ]:
# Visualize tuning improvement
fig, ax = plt.subplots(figsize=(12, 5))

model_names = list(tuning_configs.keys())
x = np.arange(len(model_names))
width = 0.35

baseline_scores = [baseline_results[n]['mean'] for n in model_names]
tuned_scores = [tuned_models[n]['best_score'] for n in model_names]

bars1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline', color='#bdc3c7', alpha=0.8)
bars2 = ax.bar(x + width/2, tuned_scores, width, label='Tuned', color='#2ecc71', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('RMSLE (lower is better)')
ax.set_title('Baseline vs Tuned Model Performance')
ax.set_xticks(x)
ax.set_xticklabels([n.split(' (')[0] for n in model_names], rotation=15)
ax.legend()

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---
# 🏆 Phase 5: Ensembling & Final Submission

Combine the tuned models using ensembling techniques to squeeze out the best possible performance. We'll try:

1. **Weighted Average** — blend predictions with inverse-RMSLE weights
2. **Stacking** — use a Ridge meta-learner on base model predictions
3. **Best individual model** — as a comparison baseline

Then select the approach with the best CV score for final submission.

## 5.1 Weighted Average Ensemble

Weight each model's contribution inversely to its RMSLE — better models get higher weights.

In [ ]:
# ── Compute inverse-RMSLE weights ──
# Exclude CART from ensemble (too weak, adds noise)
ensemble_models = {k: v for k, v in tuned_models.items() if 'CART' not in k}

scores = {name: info['best_score'] for name, info in ensemble_models.items()}
inv_scores = {name: 1.0 / score for name, score in scores.items()}
total_inv = sum(inv_scores.values())
weights = {name: inv / total_inv for name, inv in inv_scores.items()}

print('Ensemble weights (inverse-RMSLE):')
for name, w in weights.items():
    print(f'  {name:35s} weight = {w:.4f} (RMSLE: {scores[name]:.5f})')

In [ ]:
# ── Evaluate weighted average via CV ──
kf_final = KFold(n_splits=10, shuffle=True, random_state=42)

weighted_cv_scores = []

for fold_idx, (train_idx, val_idx) in enumerate(kf_final.split(X_train)):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Get weighted predictions
    blended_pred = np.zeros(len(val_idx))
    for name, info in ensemble_models.items():
        model = clone(info['model'])
        model.fit(X_fold_train, y_fold_train)
        pred = model.predict(X_fold_val)
        blended_pred += weights[name] * pred

    fold_rmse = np.sqrt(mean_squared_error(y_fold_val, blended_pred))
    weighted_cv_scores.append(fold_rmse)

weighted_mean = np.mean(weighted_cv_scores)
weighted_std = np.std(weighted_cv_scores)
print(f'\nWeighted Average Ensemble CV RMSLE: {weighted_mean:.5f} (+/- {weighted_std:.5f})')

## 5.2 Stacking Ensemble

Use the 3 strongest tuned models (RF, GBR, XGBoost) as base learners, with a **Ridge regression** meta-learner that learns the optimal combination.

In [ ]:
# ── Build stacking ensemble ──
stacking_estimators = [
    (name, clone(info['model']))
    for name, info in ensemble_models.items()
]

stacking_model = StackingRegressor(
    estimators=stacking_estimators,
    final_estimator=Ridge(alpha=10.0),
    cv=10,
    n_jobs=-1,
)

# Evaluate stacking via CV
stacking_scores = rmsle_cv(stacking_model, X_train, y_train)
stacking_mean = stacking_scores.mean()
stacking_std = stacking_scores.std()

print(f'Stacking Ensemble CV RMSLE: {stacking_mean:.5f} (+/- {stacking_std:.5f})')

## 5.3 Final Model Selection

Compare individual tuned models against ensemble methods to pick the best approach.

In [ ]:
# ── Compare all approaches ──
all_approaches = {}

# Individual tuned models
for name, info in tuned_models.items():
    all_approaches[name] = info['best_score']

# Ensembles
all_approaches['Weighted Average Ensemble'] = weighted_mean
all_approaches['Stacking Ensemble'] = stacking_mean

# Sort by score
sorted_approaches = sorted(all_approaches.items(), key=lambda x: x[1])

print('=' * 55)
print(f'{"Rank":>4s}  {"Approach":35s} {"RMSLE":>10s}')
print('=' * 55)
for rank, (name, score) in enumerate(sorted_approaches, 1):
    marker = ' 🏆' if rank == 1 else ''
    print(f'{rank:4d}  {name:35s} {score:10.5f}{marker}')
print('=' * 55)

best_approach_name = sorted_approaches[0][0]
best_approach_score = sorted_approaches[0][1]
print(f'\n🏆 Selected: {best_approach_name} (RMSLE: {best_approach_score:.5f})')

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 5))

approach_names = [x[0] for x in sorted_approaches]
approach_scores = [x[1] for x in sorted_approaches]

colors = []
for name in approach_names:
    if 'Ensemble' in name or 'Stacking' in name or 'Weighted' in name:
        colors.append('#e74c3c')  # Red for ensembles
    elif name == best_approach_name:
        colors.append('#2ecc71')  # Green for best
    else:
        colors.append('#3498db')  # Blue for individuals

bars = ax.barh(approach_names, approach_scores, color=colors, alpha=0.85)
ax.set_xlabel('RMSLE (lower is better)')
ax.set_title('All Approaches — Final Comparison')
ax.invert_yaxis()

for bar, score in zip(bars, approach_scores):
    ax.text(score + 0.001, bar.get_y() + bar.get_height()/2,
            f'{score:.5f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 5.4 Final Prediction & Submission

Retrain the best approach on the **full training set** and generate predictions for the test set.

In [ ]:
# ── Generate final predictions based on the best approach ──

if 'Stacking' in best_approach_name:
    print(f'Training final Stacking Ensemble on full data...')
    final_model = StackingRegressor(
        estimators=[
            (name, clone(info['model']))
            for name, info in ensemble_models.items()
        ],
        final_estimator=Ridge(alpha=10.0),
        cv=10,
        n_jobs=-1,
    )
    final_model.fit(X_train, y_train)
    y_pred_log = final_model.predict(X_test)

elif 'Weighted' in best_approach_name:
    print(f'Training Weighted Average Ensemble on full data...')
    y_pred_log = np.zeros(len(X_test))
    for name, info in ensemble_models.items():
        model = clone(info['model'])
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        y_pred_log += weights[name] * pred
        print(f'  {name}: weight={weights[name]:.4f}')

else:
    print(f'Training final {best_approach_name} on full data...')
    final_model = clone(tuned_models[best_approach_name]['model'])
    final_model.fit(X_train, y_train)
    y_pred_log = final_model.predict(X_test)

# Reverse log transformation
y_pred = np.expm1(y_pred_log)

# Clip any negative predictions (shouldn't happen, but safety)
y_pred = np.clip(y_pred, 0, None)

print(f'\n✅ Predictions generated for {len(y_pred)} test samples')

### Sanity Checks

In [ ]:
# ── Sanity checks ──
print('📋 Prediction sanity checks:')
print(f'  Count:   {len(y_pred)}')
print(f'  Min:     ${y_pred.min():,.0f}')
print(f'  Max:     ${y_pred.max():,.0f}')
print(f'  Mean:    ${y_pred.mean():,.0f}')
print(f'  Median:  ${np.median(y_pred):,.0f}')
print(f'  NaN:     {np.isnan(y_pred).sum()}')
print(f'  Negative: {(y_pred < 0).sum()}')

# Compare distribution to training prices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(train['SalePrice'], kde=True, ax=axes[0], color='#3498db', bins=50, label='Train (actual)')
axes[0].set_title('Training SalePrice Distribution')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].legend()

sns.histplot(y_pred, kde=True, ax=axes[1], color='#e74c3c', bins=50, label='Test (predicted)')
axes[1].set_title('Test SalePrice Distribution (Predicted)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].legend()

plt.tight_layout()
plt.show()

print('\n💡 The predicted distribution should look similar to the training distribution.')

### Generate Submission File

In [ ]:
# ── Create submission ──
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': y_pred
})

submission_path = OUTPUT_DIR + 'submission.csv'
submission.to_csv(submission_path, index=False)

print(f'📝 Submission saved to: {submission_path}')
print(f'   Rows: {len(submission)}')
print(f'   Columns: {list(submission.columns)}')
print()
print('Preview:')
submission.head(10)